In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.drop(['Unnamed: 32', 'id'], axis=1, inplace=True)

In [5]:
# train test split
x_train, x_test, y_train, y_test = train_test_split(df.drop('diagnosis', axis=1), df['diagnosis'], test_size=0.2, random_state=42)

In [6]:
# scaling
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [7]:
x_train

array([[-1.44075296, -0.43531947, -1.36208497, ...,  0.9320124 ,
         2.09724217,  1.88645014],
       [ 1.97409619,  1.73302577,  2.09167167, ...,  2.6989469 ,
         1.89116053,  2.49783848],
       [-1.39998202, -1.24962228, -1.34520926, ..., -0.97023893,
         0.59760192,  0.0578942 ],
       ...,
       [ 0.04880192, -0.55500086, -0.06512547, ..., -1.23903365,
        -0.70863864, -1.27145475],
       [-0.03896885,  0.10207345, -0.03137406, ...,  1.05001236,
         0.43432185,  1.21336207],
       [-0.54860557,  0.31327591, -0.60350155, ..., -0.61102866,
        -0.3345212 , -0.84628745]], shape=(455, 30))

In [8]:
#label encoding
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [9]:
#numpy array to pytorch tensor
x_train = torch.tensor(x_train, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
##dataset and dataloader
from torch.utils.data import DataLoader,Dataset

class customDataSet(Dataset):
    def __init__(self,features,labels):
        self.features=features
        self.labels=labels
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
            return self.features[index],self.labels[index]

In [11]:
# setting up the datasets
training_set=customDataSet(x_train,y_train)
test_set=customDataSet(x_test,y_test)

In [12]:
# setting up the dataloaders
training_loader=DataLoader(dataset=training_set,batch_size=32,shuffle=True)
testing_loader=DataLoader(dataset=test_set,batch_size=32,shuffle=True)

In [13]:
class MyNeuralNetwork(nn.Module):
    def __init__(self, num_features):
        #super(MyNeuralNetwork, self).__init__()
        super(MyNeuralNetwork, self).__init__()
        self.model=nn.Sequential(
            nn.Linear(num_features, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        output = self.model(x)
        return output

   
    
        
    

In [14]:
#initializing parameters
learning_rate = 0.001
epochs = 1000

In [15]:
loss_function = nn.BCELoss()

In [20]:
# training pipeline
model = MyNeuralNetwork(x_train.shape[1])
#optimizer
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)
#training loop
for i in range(epochs):
    
    for batch_features, batch_labels in training_loader:
        # forward pass
        y_pred = model(batch_features)
        
        # compute loss
        loss = loss_function(y_pred.view(-1), batch_labels)

        # zero the gradients after updating weights and bias
        optimizer.zero_grad()
            
        # backward pass
        loss.backward()

        #parameter update
        optimizer.step()
    
    if i % 100 == 0:
        print(f'Epoch {i}, Loss: {loss.item()}')


Epoch 0, Loss: 0.5900416970252991
Epoch 100, Loss: 3.6750087019754574e-05
Epoch 200, Loss: 5.492808486451395e-05
Epoch 300, Loss: 0.00013524555834010243
Epoch 400, Loss: 0.00012504150799941272
Epoch 500, Loss: 1.5421177579355572e-07
Epoch 600, Loss: 8.550929742057178e-09
Epoch 700, Loss: 2.4697917524463264e-06
Epoch 800, Loss: 1.5466285196996665e-10
Epoch 900, Loss: 4.013197105465552e-16


In [21]:
# eval our model
accuracyList=[]
with torch.no_grad():
    for batch_features,batch_labels in testing_loader:
        y_pred_test = model.forward(batch_features)
        y_pred_test = (y_pred_test > 0.5).float()
        accuracy = (y_pred_test.squeeze() == batch_labels).float().mean()
        accuracyList.append(accuracy)
        print(f'Test Accuracy: {accuracy.item()}')

avg_acc=sum(accuracyList)/len(accuracyList)
print("average accuracy : ",avg_acc)

Test Accuracy: 0.9375
Test Accuracy: 1.0
Test Accuracy: 0.96875
Test Accuracy: 0.9444444179534912
average accuracy :  tensor(0.9627)
